In [21]:
'''
This is a notebook to help write config files for different arg inference tools
In cases where a large contig has to be broken down into multiple differing runs
This will output a csv file with the params file for each tool
'''

'\nThis is a notebook to help write config files for different arg inference tools\nIn cases where a large contig has to be broken down into multiple differing runs\nThis will output a csv file with the params file for each tool\n'

In [22]:
import yaml
import pandas as pd
import re
import glob
import numpy as np
import os

pd.set_option('display.max_colwidth', None)

In [23]:
#DEFINE Defaults:
MAX_CONTIG_LENGTH = 5_000_000
CONTIG_OVERLAP = 300_000
MIN_CONTIG_LENGTH = 750_000
NE = 10_000

arg_inference_method = "tsinfer".lower()
basedir = "/grid/siepel/home/khalid"

SINGULARITY_IMAGE = f"{basedir}/argsortium-inference/shared/container/arg_inference_tools.sif"
OUTPUT_DIR = f"{basedir}/argsortium-outputs/2026_05_11_simulation_sets/inferred_args/{arg_inference_method}"
CONFIG_OUTPUT = f"{basedir}/argsortium-inference/{arg_inference_method}/myconfigs/2026_05_11_runs"
input_folder_base = f"{basedir}/argsortium-outputs/2026_05_11_simulation_sets/"

patterns = [
    f"{input_folder_base}/HomSap_OutOfAfrica_3G09/CEU_0_CHB_20_YRI_0/chr11_116827019_120727019_sim_seed[1-3]*params.csv",
    f"{input_folder_base}/HomSap_AmericanAdmixture_4B18/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019_sim_seed[1-3]*params.csv",
]

In [25]:
print(patterns)

['/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets//HomSap_OutOfAfrica_3G09/CEU_0_CHB_20_YRI_0/chr11_116827019_120727019_sim_seed[1-3]*params.csv', '/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets//HomSap_AmericanAdmixture_4B18/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019_sim_seed[1-3]*params.csv']


In [24]:
def sort_samples_str(samples_str):
    '''
    This function will be deprecated once the naming convention in the simulations has been updated.
    '''
    # Split into (pop, count) pairs: ["YRI", "20", "CEU", "0", "CHB", "0"]
    parts = samples_str.split("_")
    # Zip into pairs: [("YRI", "20"), ("CEU", "0"), ("CHB", "0")]
    pairs = [(parts[i], parts[i+1]) for i in range(0, len(parts), 2)]
    # Sort alphabetically by pop name and rejoin
    return "_".join([f"{pop}_{count}" for pop, count in sorted(pairs)])

def chunk_row(row):
    """Break a row into overlapping chunks if simulated_length > MAX_CONTIG_LENGTH.
    Pad chunks smaller than MIN_CONTIG_LENGTH by extending left or right."""
    if row["simulated_length"] <= MAX_CONTIG_LENGTH:
        chunks = [row.copy()]
        chunks[0]["inference_start"] = row["start"]
        chunks[0]["inference_end"] = row["end"]
        chunks[0]["inference_length"] = row["end"] - row["start"]
    else:
        step = MAX_CONTIG_LENGTH - CONTIG_OVERLAP
        chunks = []
        chunk_start = row["start"]

        while chunk_start < row["end"]:
            chunk_end = min(chunk_start + MAX_CONTIG_LENGTH, row["end"])
            new_row = row.copy()
            new_row["inference_start"] = chunk_start
            new_row["inference_end"] = chunk_end
            new_row["inference_length"] = chunk_end - chunk_start
            chunks.append(new_row)

            if chunk_end == row["end"]:
                break
            chunk_start += step

    # Pad any chunk below MIN_CONTIG_LENGTH
    for chunk in chunks:
        if chunk["inference_length"] < MIN_CONTIG_LENGTH:
            deficit = MIN_CONTIG_LENGTH - chunk["inference_length"]
            if chunk["inference_end"] == row["end"]:
                # At the end of the region — extend leftward
                chunk["inference_start"] = max(row["start"], chunk["inference_start"] - deficit)
            else:
                # At the start of the region — extend rightward
                chunk["inference_end"] = min(row["end"], chunk["inference_end"] + deficit)
            chunk["inference_length"] = chunk["inference_end"] - chunk["inference_start"]

    return chunks

In [11]:
#define all configs here
argweaver_config = {
    "singularity": SINGULARITY_IMAGE,
    "output_dir": "",
    "params_pattern": [],
    "max_contig_length": MAX_CONTIG_LENGTH,
    "start": None,
    "end": None,
    "Ne": None,
    "recomb_rate": None,
    "mcmc_samples": 4000, #defaults
    "compression": 10,
    "n_time_points": None,
    "sample_step": 20,
}

singer_config = {
    "singularity" : SINGULARITY_IMAGE,
    "output_dir" : "",
    "params_pattern" : [],
    "max_contig_length" : MAX_CONTIG_LENGTH,
    "start" : None,
    "end" : None,
    "Ne" : 2*10_000, #haploid Ne
    "mcmc_samples" : 2000, #SINGER converges quicker
    "recomb_ratio" : None,
    "thin" : 20,
    "polar" : 0.99 #use 0.99 for polarized data
    
}

relate_config = {
    "singularity" : SINGULARITY_IMAGE,
    "output_dir" : "",
    "start" : None,
    "end" : None,
    "Ne" : 2*10_000, #haploid Ne,
    "iter_start" : 0,
    "iter_end" : 50,
    "thin_every" : 5
}

tsinfer_config = {
    "singularity" : SINGULARITY_IMAGE,
    "output_dir" : "",
    "recombination_rate": 1.2e-8,
    "num_threads" : 4    
}

threads_config = {
   "singularity" : SINGULARITY_IMAGE,
    "output_dir" : "",
    "num_threads" : 4,
    "smc_n_samples" : 10, #how many chromosomes to use for smc++ .demo file
    "demography_file" : None 
}

asmc_clust_config = {
    
}

polegon_config = {
    
}

config_map = {
    "argweaver" : argweaver_config,
    "singer" : singer_config,
    "relate" : relate_config,
    "tsinfer" : tsinfer_config,
    "asmc_clust" : asmc_clust_config,
    "polegon_config" : polegon_config,
    "threads" : threads_config
}

config_to_use = config_map[arg_inference_method]

In [12]:
#find all params files and read them in to 1 pandas dataframe:
matched_params_files = sorted(
    f for pattern in patterns for f in glob.glob(pattern)
)

params_df = pd.concat(
    [pd.read_csv(f).assign(source_file=f) for f in matched_params_files],
    ignore_index=True
)
params_df["samples"] = params_df["samples"].apply(sort_samples_str)

In [13]:
grouped_df = params_df.groupby(["samples", "contig", "start", "end", "simulated_length"]).count()[["vcf_file"]]\
.rename(columns = {"vcf_file" : "number_of_seeds"}).reset_index()

chunked_df = pd.DataFrame(
    [chunk for row in grouped_df.itertuples(index=False)
     for chunk in chunk_row(row._asdict())]
).reset_index(drop=True)

chunked_df

,samples,contig,start,end,simulated_length,number_of_seeds,inference_start,inference_end,inference_length
0,ADMIX_20_AFR_0_EUR_0,chr11,116827019,120727019,3900000,3,116827019,120727019,3900000
1,CEU_0_CHB_20_YRI_0,chr11,116827019,120727019,3900000,3,116827019,120727019,3900000


In [14]:
merged_file = pd.merge(chunked_df, params_df, on = ["contig", "samples", "start", "end", "simulated_length"])
merged_file["output_dir"] = merged_file[["samples", "contig", "start", "end"]]\
.apply(lambda x : "/".join([OUTPUT_DIR, str(x[0]), "_".join([str(x[1]), str(x[2]), str(x[3])])]), axis = 1)

/tmp/ipykernel_2604339/1326930485.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  .apply(lambda x : "/".join([OUTPUT_DIR, str(x[0]), "_".join([str(x[1]), str(x[2]), str(x[3])])]), axis = 1)


In [15]:
merged_file

,samples,contig,start,end,simulated_length,number_of_seeds,inference_start,inference_end,inference_length,vcf_file,mu,recomb_map,leaves,model,seed,ancestral_fasta,source_file,output_dir
0,ADMIX_20_AFR_0_EUR_0,chr11,116827019,120727019,3900000,3,116827019,120727019,3900000,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019_sim_seed1.mutated.no_multiallelics.filtered.vcf.gz,1.290000e-08,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_84_AFR_83_EUR_83/chr11_116827019_120727019_sim_seed1.hapmap,40,AmericanAdmixture_4B18,1,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_84_AFR_83_EUR_83/chr11_116827019_120727019_sim_seed1.mutated.ancestral.fa.gz,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets//HomSap_AmericanAdmixture_4B18/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019_sim_seed1.mutated.params.csv,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/inferred_args/tsinfer/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019
1,ADMIX_20_AFR_0_EUR_0,chr11,116827019,120727019,3900000,3,116827019,120727019,3900000,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019_sim_seed2.mutated.no_multiallelics.filtered.vcf.gz,1.290000e-08,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_84_AFR_83_EUR_83/chr11_116827019_120727019_sim_seed2.hapmap,40,AmericanAdmixture_4B18,2,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_84_AFR_83_EUR_83/chr11_116827019_120727019_sim_seed2.mutated.ancestral.fa.gz,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets//HomSap_AmericanAdmixture_4B18/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019_sim_seed2.mutated.params.csv,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/inferred_args/tsinfer/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019
2,ADMIX_20_AFR_0_EUR_0,chr11,116827019,120727019,3900000,3,116827019,120727019,3900000,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019_sim_seed3.mutated.no_multiallelics.filtered.vcf.gz,1.290000e-08,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_84_AFR_83_EUR_83/chr11_116827019_120727019_sim_seed3.hapmap,40,AmericanAdmixture_4B18,3,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_84_AFR_83_EUR_83/chr11_116827019_120727019_sim_seed3.mutated.ancestral.fa.gz,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets//HomSap_AmericanAdmixture_4B18/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019_sim_seed3.mutated.params.csv,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/inferred_args/tsinfer/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019
3,CEU_0_CHB_20_YRI_0,chr11,116827019,120727019,3900000,3,116827019,120727019,3900000,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_OutOfAfrica_3G09/CEU_0_CHB_20_YRI_0/chr11_116827019_120727019_sim_seed1.mutated.no_multiallelics.filtered.vcf.gz,1.290000e-08,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_OutOfAfrica_3G09/CEU_0_CHB_125_YRI_125/chr11_116827019_120727019_sim_seed1.hapmap,40,OutOfAfrica_3G09,1,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_OutOfAfrica_3G09/CEU_0_CHB_125_YRI_125/chr11_116827019_120727019_sim_seed1.mutated.ancestral.fa.gz,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets//HomSap_OutOfAfrica_3G09/CEU_0_CHB_20_YRI_0/chr11_116827019_120727019_sim_seed1.mutated.params.csv,/grid/si

In [19]:
import csv as csv_module

MASTER_CSV_PATH = os.path.join(CONFIG_OUTPUT, "master_params.csv")
os.makedirs(CONFIG_OUTPUT, exist_ok=True)

output_rows = []
seen_uids = set()

for _, row in merged_file.iterrows():
    vcf_basename = os.path.basename(row["vcf_file"]).replace(".no_multiallelics.filtered.vcf.gz", "")
    pop_str = row["samples"]
    inf_start = int(row["inference_start"])
    inf_end   = int(row["inference_end"])

    # Append chunk window to uid when the inference window is a sub-region of the simulated contig
    if inf_start != int(row["start"]) or inf_end != int(row["end"]):
        uid = f"{pop_str}__{vcf_basename}__{inf_start}_{inf_end}"
    else:
        uid = f"{pop_str}__{vcf_basename}"

    assert uid not in seen_uids, f"Duplicate uid: {uid}"
    seen_uids.add(uid)

    output_rows.append({
        "uid":             uid,
        "vcf_file":        row["vcf_file"],
        "mu":              row["mu"],
        "recomb_map":      row["recomb_map"],
        "output_dir":      row["output_dir"],
        "ancestral_fasta": row["ancestral_fasta"],
        "contig" : row["contig"],
        "inference_start": inf_start,
        "inference_end":   inf_end,
    })

fieldnames = ["uid", "vcf_file", "mu", "recomb_map", "contig", "output_dir", "inference_start", "inference_end", "ancestral_fasta"]
with open(MASTER_CSV_PATH, "w", newline="") as f:
    writer = csv_module.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(output_rows)

print(f"Wrote {len(output_rows)} runs to {MASTER_CSV_PATH}")

Wrote 6 runs to /grid/siepel/home/khalid/argsortium-inference/tsinfer/myconfigs/2026_05_11_runs/master_params.csv


In [17]:
#for threads add demography file:
df = pd.read_csv(f"/grid/siepel/home/khalid/argsortium-inference/{arg_inference_method}/myconfigs/2026_05_11_runs/master_params.csv")
df["demo_file"] = df["recomb_map"].apply(lambda x : x.split(".hapmap")[0] + ".mutated.no_multiallelics.filtered.demo")
df.to_csv(MASTER_CSV_PATH, index = False)

In [20]:
pd.read_csv(f"/grid/siepel/home/khalid/argsortium-inference/{arg_inference_method}/myconfigs/2026_05_11_runs/master_params.csv")

,uid,vcf_file,mu,recomb_map,contig,output_dir,inference_start,inference_end,ancestral_fasta
0,ADMIX_20_AFR_0_EUR_0__chr11_116827019_120727019_sim_seed1.mutated,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019_sim_seed1.mutated.no_multiallelics.filtered.vcf.gz,1.290000e-08,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_84_AFR_83_EUR_83/chr11_116827019_120727019_sim_seed1.hapmap,chr11,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/inferred_args/tsinfer/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019,116827019,120727019,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_84_AFR_83_EUR_83/chr11_116827019_120727019_sim_seed1.mutated.ancestral.fa.gz
1,ADMIX_20_AFR_0_EUR_0__chr11_116827019_120727019_sim_seed2.mutated,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019_sim_seed2.mutated.no_multiallelics.filtered.vcf.gz,1.290000e-08,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_84_AFR_83_EUR_83/chr11_116827019_120727019_sim_seed2.hapmap,chr11,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/inferred_args/tsinfer/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019,116827019,120727019,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_84_AFR_83_EUR_83/chr11_116827019_120727019_sim_seed2.mutated.ancestral.fa.gz
2,ADMIX_20_AFR_0_EUR_0__chr11_116827019_120727019_sim_seed3.mutated,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019_sim_seed3.mutated.no_multiallelics.filtered.vcf.gz,1.290000e-08,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_84_AFR_83_EUR_83/chr11_116827019_120727019_sim_seed3.hapmap,chr11,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/inferred_args/tsinfer/ADMIX_20_AFR_0_EUR_0/chr11_116827019_120727019,116827019,120727019,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_AmericanAdmixture_4B18/ADMIX_84_AFR_83_EUR_83/chr11_116827019_120727019_sim_seed3.mutated.ancestral.fa.gz
3,CEU_0_CHB_20_YRI_0__chr11_116827019_120727019_sim_seed1.mutated,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_OutOfAfrica_3G09/CEU_0_CHB_20_YRI_0/chr11_116827019_120727019_sim_seed1.mutated.no_multiallelics.filtered.vcf.gz,1.290000e-08,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_OutOfAfrica_3G09/CEU_0_CHB_125_YRI_125/chr11_116827019_120727019_sim_seed1.hapmap,chr11,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/inferred_args/tsinfer/CEU_0_CHB_20_YRI_0/chr11_116827019_120727019,116827019,120727019,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_OutOfAfrica_3G09/CEU_0_CHB_125_YRI_125/chr11_116827019_120727019_sim_seed1.mutated.ancestral.fa.gz
4,CEU_0_CHB_20_YRI_0__chr11_116827019_120727019_sim_seed2.mutated,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_OutOfAfrica_3G09/CEU_0_CHB_20_YRI_0/chr11_116827019_120727019_sim_seed2.mutated.no_multiallelics.filtered.vcf.gz,1.290000e-08,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_OutOfAfrica_3G09/CEU_0_CHB_125_YRI_125/chr11_116827019_120727019_sim_seed2.hapmap,chr11,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/inferred_args/tsinfer/CEU_0_CHB_20_YRI_0/chr11_116827019_120727019,116827019,120727019,/grid/siepel/home/khalid/argsortium-outputs/2026_05_11_simulation_sets/HomSap_OutOfAfrica_3G09/CEU_0_CHB_125_YRI_125/chr11_116827019_120727019_sim_seed2